# 02.5 Binding: Exercises

## Contents

**Tools:** CADET-Process (Python).

**Python used:** Model setup, parameter changes and concentration-array comparisons.

**Practises:** setting up `Linear` and `Langmuir` binding models; linear retention, Langmuir capacity and binding kinetics in a column.

1. Linear binding and retention
2. Loading and capacity
3. Binding kinetics (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Keep transport fixed and compare protein retention with a nonbinding tracer. Exercises marked **optional** extend the main tasks and can be skipped.

```{note}
We use Python in this exercise to practise setting up a process. If you get stuck, use the [GUI Workbench](../../99%20Misc/99.2%20GUI%20Workbench/99.2a_gui_workbench_lesson.ipynb) to build the setup and copy its Python code with **Show code**. Check the parameters against this exercise before running it.
```

## Starter: Linear binding

```{note}
The supplied 10 × 2 cm column has porosity 0.4, dispersion `1e-7` m²/s and flow 6 mL/min. Feed concentrations are `[1, 1, 1]` mol/m³ for Tracer, Protein A and Protein B. The column has no binding model yet; it is added in Exercise 1.
```

```{tip}
Run the cell below once with **Shift+Enter** to set up the supplied model for this exercise.
```

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from CADETProcess.processModel import (
    ComponentSystem, FlowSheet, Process, Inlet, Outlet,
    LumpedRateModelWithoutPores, Linear, Langmuir,
)
from CADETProcess.simulator import Cadet
import numpy as np

linear_component_system = ComponentSystem()
linear_sheet = FlowSheet(linear_component_system, "flow_sheet")
linear_process = Process(linear_sheet, "linear")

linear_component_system.add_component("Tracer")
linear_component_system.add_component("Protein A")
linear_component_system.add_component("Protein B")
inlet = Inlet(linear_component_system, "inlet")
inlet.c = [1, 1, 1]
inlet.flow_rate = 1e-7
column = LumpedRateModelWithoutPores(linear_component_system, "column")
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7
outlet = Outlet(linear_component_system, "outlet")
linear_sheet.add_unit(inlet)
linear_sheet.add_unit(column)
linear_sheet.add_unit(outlet)
linear_sheet.add_connection(inlet, column)
linear_sheet.add_connection(column, outlet)
linear_process.cycle_time = 1200

## Exercise 1: retention

Create a `Linear` binding model for `linear_component_system`, set the values below and assign it to `column.binding_model`. Lists follow **Tracer, Protein A, Protein B**; the tracer does not bind.

| Attribute | Value |
|---|---|
| `is_kinetic` | `False` (rapid equilibrium) |
| `adsorption_rate` | `[0, 2, 1]` |
| `desorption_rate` | `[1, 1, 1]` |

In [ ]:
# Linear binding model


Use $t_R=[\varepsilon_t+(1-\varepsilon_t)K]V/Q$ to predict the breakthrough order. Here $K=k_a/k_d$. Calculate the retention times in **seconds** in the answer cell.

In [ ]:
# Predicted retention times


The supplied simulation plots the original breakthrough. `column.binding_model = binding_model` in your answer above attached the binding model to the column.

In [ ]:
# Simulate and plot the supplied linear-binding column.
linear_baseline = Cadet().simulate(linear_process)
_ = linear_baseline.solution.outlet.outlet.plot()
plt.show()

Change Protein A's adsorption rate from **2 to 4**, keeping the other components at **0 and 1**, through `linear_process.flow_sheet.column.binding_model.adsorption_rate`. Which curves change?

In [ ]:
# Change the adsorption rate of Protein A


In [ ]:
# Simulate the changed binding parameters and plot the outlet.
linear_changed = Cadet().simulate(linear_process)
_ = linear_changed.solution.outlet.outlet.plot()
plt.show()

```{dropdown} Answer
Tracer arrives first (about 126 s), Protein B next (314 s), Protein A last (503 s). A stronger linear equilibrium coefficient delays a protein without changing the tracer. Assigning protein retention to extra liquid volume would also delay the tracer and would misinterpret binding as porosity.
```

## Starter: Langmuir binding

```{note}
The next column has the same transport settings and one component, Protein. Feed is 1 mol/m³; duration is 4000 s. Its binding model is added in Exercise 2.
```

```{tip}
Run the cell below once with **Shift+Enter** to set up the supplied model for this exercise.
```

In [ ]:
from CADETProcess.processModel import (
    ComponentSystem, FlowSheet, Process, Inlet, Outlet,
    LumpedRateModelWithoutPores, Linear, Langmuir,
)
from CADETProcess.simulator import Cadet
import numpy as np

capacity_component_system = ComponentSystem()
capacity_sheet = FlowSheet(capacity_component_system, "flow_sheet")
capacity_process = Process(capacity_sheet, "capacity")

capacity_component_system.add_component("Protein")
inlet = Inlet(capacity_component_system, "inlet")
inlet.c = [1]
inlet.flow_rate = 1e-7
column = LumpedRateModelWithoutPores(capacity_component_system, "column")
column.length = 0.1
column.diameter = 0.02
column.total_porosity = 0.4
column.axial_dispersion = 1e-7
outlet = Outlet(capacity_component_system, "outlet")
capacity_sheet.add_unit(inlet)
capacity_sheet.add_unit(column)
capacity_sheet.add_unit(outlet)
capacity_sheet.add_connection(inlet, column)
capacity_sheet.add_connection(column, outlet)
capacity_process.cycle_time = 4000

## Exercise 2: loading and capacity

Create a `Langmuir` binding model for `capacity_component_system` with the values below and assign it to `column.binding_model`.

| Attribute | Value |
|---|---|
| `is_kinetic` | `False` (rapid equilibrium) |
| `adsorption_rate` | `[2]` |
| `desorption_rate` | `[1]` |
| `capacity` | `[10]` mol/m³ of solid |

In [ ]:
# Langmuir binding model


Predict the breakthrough order for feeds of **0.2, 1 and 5 mol/m³**, then run the supplied comparison code. Compare $c_\mathrm{out}/c_\mathrm{feed}$. Which front arrives first?
Keep geometry, flow and porosity fixed while changing binding and feed conditions.

In [ ]:
# Run each feed concentration separately and keep its outlet signal.
capacity_process.flow_sheet.inlet.c = [0.2]
low_feed = Cadet().simulate(capacity_process).solution.outlet.outlet

capacity_process.flow_sheet.inlet.c = [1]
reference_feed = Cadet().simulate(capacity_process).solution.outlet.outlet

capacity_process.flow_sheet.inlet.c = [5]
high_feed = Cadet().simulate(capacity_process).solution.outlet.outlet
# Restore the 1 mol/m³ feed for the capacity comparison.
capacity_process.flow_sheet.inlet.c = [1]

In [ ]:
# Overlay the three stored signals, each divided by its own feed concentration.
fig, ax = plt.subplots(figsize=(8, 3))
# solution[:, 0]: the only component, Protein
ax.plot(low_feed.time, low_feed.solution[:, 0] / 0.2, label="feed 0.2")
ax.plot(reference_feed.time, reference_feed.solution[:, 0] / 1, label="feed 1")
ax.plot(high_feed.time, high_feed.solution[:, 0] / 5, label="feed 5")
ax.set(xlabel="Time / s", ylabel="Outlet / feed")
ax.legend()
plt.show()

At feed **1 mol/m³**, halve capacity from **10 to 5 mol/m³** through `capacity_process.flow_sheet.column.binding_model.capacity`. Explain the change in breakthrough.

In [ ]:
# Halve the capacity


In [ ]:
# Compare the original and reduced capacities at feed 1 mol/m³.
reduced_capacity = Cadet().simulate(capacity_process).solution.outlet.outlet
fig, ax = plt.subplots(figsize=(8, 3))
# solution[:, 0]: the only component, Protein
ax.plot(reference_feed.time, reference_feed.solution[:, 0], label="capacity 10")
ax.plot(reduced_capacity.time, reduced_capacity.solution[:, 0], label="capacity 5")
ax.set(xlabel="Time / s", ylabel="Concentration / mol/m³")
ax.legend()
plt.show()

```{dropdown} Interpretation
At high feed, the finite sites fill sooner and the normalized front arrives earlier. Reducing capacity also advances breakthrough. Affinity, capacity and transport can all affect retention, so characterize transport first and use several loading conditions to distinguish binding parameters.
```

## Exercise 3 (optional): binding kinetics

At feed 1 and capacity 10, switch to kinetic binding. Compare adsorption/desorption rates `(2, 1)` and `(0.02, 0.01)`. Their ratio is the same; must their breakthrough curves match? Answer before simulating, then check with the simulation.

In [ ]:
# Restore the feed and capacity, then simulate the faster binding rates.


```{dropdown} Answer
No. Both rate pairs have the same ratio, so they describe the same equilibrium isotherm. With rates `(2, 1)`, binding is fast compared with transport and the curve is close to the equilibrium result. With rates `(0.02, 0.01)`, binding cannot keep up with the flow: protein breaks through earlier and the front is more gradual. Equilibrium data alone cannot distinguish these cases.
```

Tracer measurements fix transport; protein loading constrains binding. Next, chromatography combines binding with the salt gradient. The characterization workbench later links these fits.

**Next lesson:** [**03.1 Simple Chromatography**](../../03%20Process%20Simulation/03.1%20Simple%20Chromatography/03.1a_simple_chromatography_lesson.ipynb)